In [2]:
from resolver import Resolver

resolver = Resolver()

In [4]:
sample = resolver.rescued_data.sample(20)
sample

,dataset,dataset_id,status,url,source_website,organization,agency,download_date,size,maintainer,download_location,file_type,notes,metadata_available,metadata_url
1505,Pulmonary evaluation of whole-body inhalation ...,1710,Finished,https://data.cdc.gov/National-Institute-for-Oc...,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,<NA>,<NA>,DL,https://www.datalumos.org/datalumos/project/23...,<NA>,<NA>,<NA>,<NA>
966,State Summaries_North Dakota,1123,Finished,https://www.data.va.gov/stories/s/5ezy-r98d,data.va.gov,Office of Information and Technology - IT Oper...,Department of Veterans Affairs,2025-04-24,0.0,"DRP, DL",https://www.datalumos.org/datalumos/project/22...,"CSV, PDF",<NA>,<NA>,<NA>
4470,"Forest structure, regeneration, and fuels in u...",6651,Finished,https://www.fs.usda.gov/rds/archive/catalog/RD...,fs.usda.gov,US Forest Service,U.S. Department of Agriculture,2026-06-05,0.0005,"DRP, DL",https://www.datalumos.org/datalumos/project/24...,"PDF, ZIP",<NA>,yes,<NA>
4657,Tree and shrub measurements in Stanislaus Nati...,6838,Finished,https://www.fs.usda.gov/rds/archive/catalog/RD...,fs.usda.gov,US Forest Service,U.S. Department of Agriculture,2026-06-05,0.0003,"DRP, DL",https://www.datalumos.org/datalumos/project/24...,"PDF, ZIP",<NA>,yes,<NA>
1412,Opioid Treatment Program Providers,1569,Finished,https://data.cms.gov/provider-characteristics/...,data.cms.gov,Centers for Medicare and Medicaid Services (CMS),Department of Health and Human Services,2025-04-30,<NA>,DL,https://www.datalumos.org/datalumos/project/23...,"CSV, PDF",The Opioid Treatment Program (OTP) Providers d...,<NA>,<NA>
957,State Summaries_Ohio,1114,Finished,https://www.data.va.gov/stories/s/98bg-sm56,data.va.gov,Office of Information and Technology - IT Oper...,Department of Veterans Affairs,2025-04-24,0.0,"DRP, DL",https://www.datalumos.org/datalumos/project/22...,CSV,<NA>,<NA>,<NA>
3600,USDA - Web Log Analysis Dashboard Dataset,5676,Finished,https://agdatacommons.nal.usda.gov/articles/da...,agdatacommons.nal.usda.gov,National Agricultural Library,U.S. Department of Agriculture,2026-06-27,0.0001,"DRP, DL",https://www.datalumos.org/datalumos/project/25...,"CSV, HTML, JSON",<NA>,yes,http://web.archive.org/web/20250915102451/http...
3444,Genes of viral origin in the Cotesia vestalis ...,5509,Finished,https://agdatacommons.nal.usda.gov/articles/da...,agdatacommons.nal.usda.gov,National Agricultural Library,U.S. Department of Agriculture,2026-06-26,0.0001,"DRP, DL",https://www.datalumos.org/datalumos/project/25...,"GZ, HTML, JSON",<NA>,yes,http://web.archive.org/web/20251017044601/http...
3321,USDA Agricultural Research Service- Available ...,5385,Finished,https://agdatacommons.nal.usda.gov/articles/da...,agdatacommons.nal.usda.gov,National Agricultural Library,U.S. Department of Agriculture,2026-06-26,0.0011,"DRP, DL",https://www.datalumos.org/datalumos/project/25...,"CSV, HTML, JSON, PDF",<NA>,yes,http://web.archive.org/web/20251113234658/http...
1078,Temporal Viral Viability Data from Avian Influ...,1236,Finished,https://data.usgs.gov/datacatalog/data/USGS:AS...,data.usgs.gov,U.S. Geological Survey,Department of the Interior,2025-04-18,0.004,"DRP, DL",https://www.datalumos.org/datalumos/project/23...,"CSV, XML, HTML",<NA>,<NA>,<NA>


In [13]:
import asyncio

import httpx
import pandas as pd


async def get_status(client: httpx.AsyncClient, url: str) -> int | str:
    try:
        response = await client.head(url, follow_redirects=True)
        return response.status_code
    except httpx.TimeoutException:
        return "timeout"
    except httpx.HTTPError as e:
        return f"error: {e}"


async def get_statuses() -> list[int | str]:
    timeout = httpx.Timeout(20.0, connect=20.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        return await asyncio.gather(*(get_status(client, url) for url in sample["url"]))


statuses = await get_statuses()
sample["status"] = statuses


pd.set_option("display.max_colwidth", 200)
sample[["url", "status"]]


,url,status
1505,https://data.cdc.gov/National-Institute-for-Occupational-Safety-and-Hea/Pulmonary-evaluation-of-whole-body-inhalation-expo/235m-gsry,200
966,https://www.data.va.gov/stories/s/5ezy-r98d,200
4470,https://www.fs.usda.gov/rds/archive/catalog/RDS-2019-0004,200
4657,https://www.fs.usda.gov/rds/archive/catalog/RDS-2017-0045,200
1412,https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/opioid-treatment-program-providers,200
957,https://www.data.va.gov/stories/s/98bg-sm56,200
3600,https://agdatacommons.nal.usda.gov/articles/dataset/USDA_-_Web_Log_Analysis_Dashboard_Dataset/24853317,202
3444,https://agdatacommons.nal.usda.gov/articles/dataset/Genes_of_viral_origin_in_the_Cotesia_vestalis_genome/24664809,202
3321,https://agdatacommons.nal.usda.gov/articles/dataset/USDA_Agricultural_Research_Service-_Available_Biological_Materials/24663015,202
1078,https://data.usgs.gov/datacatalog/data/USGS:ASC402,405


In [14]:
sample[sample["status"] != 200][["url", "status"]]

,url,status
3600,https://agdatacommons.nal.usda.gov/articles/dataset/USDA_-_Web_Log_Analysis_Dashboard_Dataset/24853317,202
3444,https://agdatacommons.nal.usda.gov/articles/dataset/Genes_of_viral_origin_in_the_Cotesia_vestalis_genome/24664809,202
3321,https://agdatacommons.nal.usda.gov/articles/dataset/USDA_Agricultural_Research_Service-_Available_Biological_Materials/24663015,202
1078,https://data.usgs.gov/datacatalog/data/USGS:ASC402,405
111,https://www.huduser.gov/portal/datasets/feasibility-natl-db.html,202
1370,https://maps2.dcgis.dc.gov/dcgis/rest/services/FEEDS/MPD/FeatureServer/9,404
3268,https://agdatacommons.nal.usda.gov/articles/dataset/Agrilus_planipennis_genome_annotations_v0_5_3/24662091,202
1570,https://hifld-geoplatform.hub.arcgis.com/,404
1562,https://hifld-geoplatform.hub.arcgis.com/,404
